In [1]:
import os
import re
import glob
from IPython.display import display
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
#import japanize_matplotlib
import seaborn as sns 
import unicodedata
import yaml



In [2]:
# configの読み込み
CONFIG_FILE = '../configs/config.yaml'
with open(CONFIG_FILE, encoding="utf-8") as file:
    yml = yaml.safe_load(file)

In [3]:
DIR_INPUT = yml["SETTING"]["DIR_INPUT"]
DIR_INTERIM = yml["SETTING"]["DIR_INTERIM"]
DIR_FEATURE = yml["SETTING"]["DIR_FEATURE"]
DIR_FIGURE = yml["SETTING"]["DIR_FIGURE"]
DIR_HOME = yml["SETTING"]["DIR_HOME"]
DIR_LOG = yml["SETTING"]["DIR_LOG"]
FILE_NAME_STATION = yml["SETTING"]["FILE_NAME_STATION"]
FILE_NAME_STATUS = yml["SETTING"]["FILE_NAME_STATUS"]
FILE_NAME_TRIP = yml["SETTING"]["FILE_NAME_TRIP"]
FILE_NAME_WEATHER = yml["SETTING"]["FILE_NAME_WEATHER"]

# データ読み込み

In [27]:
dict_dtype_station = {
    "station_id": "int64",
    "lat": "float64",
    "long": "float64",
    "dock_count": "int64",
    "city": "object",
    "installation_date": "object",
}
dict_dtype_status = {
    "id": "int64",
    "year": "int64",
    "month": "int64",
    "day": "int64",
    "hour": "int64",
    "station_id": "int64",
    "bikes_available": "float64",
    "predict": "int64",
}
dict_dtype_trip = {
    "trip_id": "int64",
    "duration": "int64",
    "start_date": "object",
    "start_station_id": "int64",
    "end_date": "object",
    "end_station_id": "int64",
    "bike_id": "int64",
    "subscription_type": "object",
}
dict_dtype_weather = {
    "date": "object",
    "max_temperature": "int64",
    "mean_temperature": "int64",
    "min_temperature": "int64",
    "max_dew_point": "int64",
    "mean_dew_point": "int64",
    "min_dew_point": "int64",
    "max_humidity": "int64",
    "mean_humidity": "int64",
    "min_humidity": "int64",
    "max_sea_level_pressure": "float64",
    "mean_sea_level_pressure": "float64",
    "min_sea_level_pressure": "float64",
    "max_visibility": "int64",
    "mean_visibility": "int64",
    "min_visibility": "int64",
    "max_wind_Speed": "int64",
    "mean_wind_speed": "int64",
    "precipitation": "float64",
    "cloud_cover": "int64",
    "events": "object",
    "wind_dir_degrees": "int64",
}

In [28]:
df_station = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_STATION), dtype=dict_dtype_station)
df_status = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_STATUS), dtype=dict_dtype_status)
df_trip = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_TRIP), dtype=dict_dtype_trip)
df_weather = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_WEATHER), dtype=dict_dtype_weather)

# 前処理

In [35]:
def convert_datetime(df_station, df_status, df_trip, df_weather):
    df_station["installation_date"] = pd.to_datetime(df_station["installation_date"], format='%m/%d/%Y').dt.date
    df_status["datetime"] = pd.to_datetime(df_status[["year", "month", "day", "hour"]])
    df_trip["start_date"] = pd.to_datetime(df_trip["start_date"], format='%m/%d/%Y %H:%M')
    df_trip["end_date"] = pd.to_datetime(df_trip["end_date"], format='%m/%d/%Y %H:%M')
    df_weather["date"] = pd.to_datetime(df_weather["date"], format='%Y-%m-%d').dt.date

    return df_station, df_status, df_trip, df_weather

In [36]:
df_station, df_status, df_trip, df_weather = convert_datetime(df_station, df_status, df_trip, df_weather)

## データ出力

In [37]:
df_station.to_pickle(os.path.join(DIR_INTERIM, "df_prep_station.pkl"))
df_status.to_pickle(os.path.join(DIR_INTERIM, "df_prep_status.pkl"))
df_trip.to_pickle(os.path.join(DIR_INTERIM, "df_prep_trip.pkl"))
df_weather.to_pickle(os.path.join(DIR_INTERIM, "df_prep_weather.pkl"))

# 特徴量作成

In [38]:
from pathlib import Path
from abc import ABCMeta, abstractmethod

In [39]:
from time import time

def decorate(s: str, decoration=None):
    if decoration is None:
        decoration = '★' * 20
        
    return ' '.join([decoration, str(s), decoration])

class Timer:
    def __init__(self, logger=None, format_str='{:.3f}[s]', prefix=None, suffix=None, sep=' ', verbose=0):

        if prefix: format_str = str(prefix) + sep + format_str
        if suffix: format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None
        self.verbose = verbose

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        if self.verbose is None:
            return
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

In [41]:
class AbstractBaseBlock(metaclass=ABCMeta):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        self.use_cache = use_cache
        self.name = self.__class__.__name__
        self.cache_dir = Path(DIR_FEATURE)
        self.logger = logger
        self.seve_cache = save_cache
        self.use_cols = None
        
    # 内部状態の更新
    def fit(self, df_input: pd.DataFrame, y=None):
        pass
    
    # 変換処理
    @abstractmethod
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError()
    
    # 内部状態の更新と変換処理をまとめて行う
    def fit_transform(self, X: pd.DataFrame, y=None):
        self.fit(X, y)
        return self.transform(X)
    
    # 特徴量生成処理
    def create_feature(self, X, y=None, test=False, limit_date="yyyy-mm-dd") -> pd.DataFrame:

        # クラス名.pkl
        file_name = os.path.join(self.cache_dir, limit_date, f"{self.name}.pkl")

        # キャッシュを使う & ファイルがあるなら読み出し
        if os.path.isfile(str(file_name)) and self.use_cache:
            return pd.read_pickle(file_name)
        
        # 変換処理を実行
        else:
            # trainの場合
            if not test:
                feature = self.fit_transform(X, y)
            # testの場合
            else:
                feature = self.transform(X)
            # 保存する場合
            if self.seve_cache:
                feature.to_pickle(file_name)

            return feature

In [9]:
# そのままの特徴量を返すBlock

In [ ]:
# 静的特徴量
# 動的特徴量

In [209]:
class AsIsNumetricBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            'CityTier', 
            'DurationOfPitch',
            'NumberOfPersonVisiting',
            'NumberOfFollowups', 
            'PreferredPropertyStar', 
            'NumberOfTrips',
            'Passport', 
            'PitchSatisfactionScore',
            'MonthlyIncome', 
            'ProdTaken' # 目的関数
        ]
        self.key_col = ["id"]
        self.map_count = None

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out = df_input[self.key_col + self.use_cols]
        return df_out

class AsIsCategoryBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            'Age', 
            'TypeofContact',
            'Occupation',
            'Gender',
            'ProductPitched',
            'Designation', 
            'customer_info', 
            'marry', 
            'car', 
            'child',
        ]
        self.key_col = ["id"]
        self.map_count = None

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out = df_input[self.key_col + self.use_cols]
        return df_out

In [73]:
class CountEncodingBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            "TypeofContact", "CityTier", "Occupation", "ProductPitched",
            "Designation", "marry", "car"
        ]
        self.key_col = ["id"]
        self.map_count = None

    def fit(self, df_input, y=None):
        # カラムごとにマッピング表を作成
        self.map_count = {}
        for col in self.use_cols:
            self.map_count[col] = df_input[col].fillna("NA").value_counts()

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        for col in self.use_cols:
            df_out[col] = df_input[col].fillna("NA").map(self.map_count[col]).astype(int)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('CE_')], axis=1)
        return df_result
        
class TargetEncodingBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            "TypeofContact", "CityTier", "Occupation", "ProductPitched",
            "Designation", "marry", "car"
        ]
        self.key_col = ["id"]
        self.map_target = None

    def fit(self, df_input, y):
        # カラムごとにマッピング表を作成
        self.map_target = {}
        for col in self.use_cols:
            self.map_target[col] = df_input.groupby(col)[TARGET_COL].mean()

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        for col in self.use_cols:
            df_out[col] = df_input[col].map(self.map_target[col]).astype(float)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('TE_')], axis=1)
        return df_result
    
# 大分類を作成するクラス
class BigCategoryBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = []
        self.key_col = ["id"]
        self.map_target = None

    def fit(self, df_input, y=None):
        pass

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        df_out["Age"] = df_input["Age"].apply(lambda x: self.replace_sai_to_dai(x) if not pd.isna(x) else x)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('BC_')], axis=1)
        return df_result
    
    def replace_sai_to_dai(self, age):
        first_char = age[0]
        if first_char == "1":
            result = "10代"
        elif first_char == "2":
            result = "20代"
        elif first_char == "3":
            result = "30代"
        elif first_char == "4":
            result = "40代"
        elif first_char == "5":
            result = "50代"
        elif first_char == "6":
            result = "60代"
        return result
    
# 大分類を作成するクラス
class BigCategoryBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = []
        self.key_col = ["id"]
        self.map_target = None

    def fit(self, df_input, y=None):
        pass

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        df_out["Age"] = df_input["Age"].apply(lambda x: self.replace_sai_to_dai(x) if not pd.isna(x) else x)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('BC_')], axis=1)
        return df_result
    
    def replace_sai_to_dai(self, age):
        first_char = age[0]
        if first_char == "1":
            result = "10代"
        elif first_char == "2":
            result = "20代"
        elif first_char == "3":
            result = "30代"
        elif first_char == "4":
            result = "40代"
        elif first_char == "5":
            result = "50代"
        elif first_char == "6":
            result = "60代"
        return result

In [75]:
def run_blocks(df_input, feature_blocks, y=None, test=False):
    df_out = None
    
    print(decorate('start run blocks...'))

    with Timer(prefix='run test={}'.format(test)):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature(df_input, y=y, test=test)
            assert len(df_input) == len(feature), block
            if df_out is None:
                df_out = feature
            else:
                df_out = pd.merge(df_out, feature, on=["id"], how="left")

    return df_out

In [210]:
feature_blocks = [
    *[AsIsCategoryBlock(use_cache=False, save_cache=True, logger=None)],
    *[AsIsNumetricBlock(use_cache=True, save_cache=True, logger=None)],
    *[CountEncodingBlock(use_cache=True, save_cache=True, logger=None)],
    *[TargetEncodingBlock(use_cache=True, save_cache=True, logger=None)],
    *[BigCategoryBlock(use_cache=True, save_cache=True, logger=None)],
]

In [211]:
df_preprocessed = pd.read_pickle(os.path.join(DIR_INTERIM, "preprocessed.pkl"))

In [212]:
df_out = run_blocks(df_preprocessed, feature_blocks, y=None, test=False)

★★★★★★★★★★★★★★★★★★★★ start run blocks... ★★★★★★★★★★★★★★★★★★★★
	- <__main__.AsIsCategoryBlock object at 0x7fd57ad0a400> 0.034[s]
	- <__main__.AsIsNumetricBlock object at 0x7fd57ad0a8b0> 0.004[s]
	- <__main__.CountEncodingBlock object at 0x7fd57ad0a8e0> 0.002[s]
	- <__main__.TargetEncodingBlock object at 0x7fd57ad0a940> 0.001[s]
	- <__main__.BigCategoryBlock object at 0x7fd57ad0ac70> 0.001[s]
run test=False 0.069[s]


In [4]:
# 特徴量生成

In [4]:
df_status = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_status.pkl"))

In [5]:
df_status_feat = df_status.copy()
df_status_feat["weekdays"] = df_status["datetime"].dt.weekday

In [6]:
#これから細かい前処理をするためにmain_dfを作成
main_df = df_status_feat[["station_id", "datetime", "bikes_available", "predict", "weekdays"]]
main_df.head()

,station_id,datetime,bikes_available,predict,weekdays
0,0,2013-09-01 00:00:00,11.0,0,6
1,0,2013-09-01 01:00:00,11.0,0,6
2,0,2013-09-01 02:00:00,11.0,0,6
3,0,2013-09-01 03:00:00,11.0,0,6
4,0,2013-09-01 04:00:00,11.0,0,6


In [7]:
#各ステーション毎に、欠損値を後の値で埋める
main_df_new = pd.DataFrame()
for station_id in main_df["station_id"].unique().tolist():
    temp_df = main_df[main_df["station_id"]==station_id]
    temp_df = temp_df.fillna(method="bfill")
    main_df_new = pd.concat([main_df_new,temp_df])

print(main_df_new.isnull().sum())

/var/folders/qk/pwcsrt352q5ds79rrtgfbtrh0000gn/T/ipykernel_17211/1622897470.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp_df = temp_df.fillna(method="bfill")


station_id         0
datetime           0
bikes_available    0
predict            0
weekdays           0
dtype: int64


In [16]:
#データセットを時系列に並び替える
main_df_new = main_df_new.sort_values(["datetime","station_id"],ascending=True).reset_index(drop=True)
#学習用データセット
main_df_new

,station_id,datetime,bikes_available,predict,weekdays
0,0,2013-09-01 00:00:00,11.0,0,6
1,1,2013-09-01 00:00:00,8.0,0,6
2,2,2013-09-01 00:00:00,5.0,0,6
3,3,2013-09-01 00:00:00,9.0,0,6
4,4,2013-09-01 00:00:00,8.0,0,6
...,...,...,...,...,...
1226395,65,2015-08-31 23:00:00,13.0,0,0
1226396,66,2015-08-31 23:00:00,7.0,0,0
1226397,67,2015-08-31 23:00:00,6.0,0,0
1226398,68,2015-08-31 23:00:00,5.0,0,0


In [25]:
def create_valid_flag(df_result):
    df_ = df_result.copy()
    df_ = df_[df_["predict"] != 0]
    df_["date"] = df_["datetime"].dt.date
    df_ = df_[["date","predict"]].drop_duplicates()
    df_["diff"] = df_["date"].diff().dt.days
    df_["yesterday"] = df_["date"].apply(lambda x: x - pd.Timedelta(days=1))
    list_valid_date = df_[df_["diff"] >= 4.0]["yesterday"].tolist()
    df_result_ = df_result.copy()
    # dateがlist_valid_dateに含まれている場合はpredictを2にする
    df_result_["predict"] = df_result_.apply(lambda x: 2 if (x["datetime"].date() in list_valid_date) & (x["datetime"].hour!=0) else x["predict"], axis=1)
    return df_result_

In [26]:
past_2_len = 0
while True:
    main_df_new = create_valid_flag(main_df_new)
    current_2_len = len(main_df_new[main_df_new["predict"] == 2])
    if current_2_len == past_2_len:
        break
    past_2_len = current_2_len

In [28]:
main_df_new.groupby("predict").size()

predict
0    928550
1    193200
2    104650
dtype: int64

In [29]:
#学習用データセット
main_df_new

,station_id,datetime,bikes_available,predict,weekdays
0,0,2013-09-01 00:00:00,11.0,0,6
1,1,2013-09-01 00:00:00,8.0,0,6
2,2,2013-09-01 00:00:00,5.0,0,6
3,3,2013-09-01 00:00:00,9.0,0,6
4,4,2013-09-01 00:00:00,8.0,0,6
...,...,...,...,...,...
1226395,65,2015-08-31 23:00:00,13.0,0,0
1226396,66,2015-08-31 23:00:00,7.0,0,0
1226397,67,2015-08-31 23:00:00,6.0,0,0
1226398,68,2015-08-31 23:00:00,5.0,0,0


# モデル構築

In [30]:
# 自作モジュールの読み込み
import sys
sys.path.append(DIR_HOME)
from src.runner import TimeseriesModelRunner
from src.model_LSTM_mult import model_LSTM_mult
from datetime import datetime
from src.util import Logger


In [160]:
import importlib
from src import runner
from src import model_LSTM_mult
from src import model

# runnerモジュールをリロード
importlib.reload(runner)
importlib.reload(model_LSTM_mult)
importlib.reload(model)

# Runnerクラスを再インポート
from src.runner import TimeseriesModelRunner
from src.model_LSTM_mult import model_LSTM_mult
from src.model import Model

In [107]:
def get_run_name(model_type):
    """run名の作成
    """
    run_name = model_type
    suffix = '_' + datetime.now().strftime("%Y%m%d%H%M")
    run_name = run_name + suffix
    return run_name

In [108]:
logger = Logger(path=DIR_LOG)

In [171]:
run_name = get_run_name(model_type="lstm_mult")
run_name

'lstm_mult_202410082220'

In [172]:
memo = "LSTMのマルチステップ予測"
model_params = {
    "key_cols": ["datetime", "station_id"],
    "target_col": "bikes_available",
    "seq_length": 24,
    "n_steps": 23,
    "batch_size": 64,
    "shuffle": True,
    "hidden_size": 64,
    "num_epochs": 10,
    "learning_rate": 0.001,
}
run_setting = {
    'calc_shap': False,     # shap値を計算するか否か
    'save_train_pred': False,    # trainデータに対する予測値を保存するか否か(閾値の最適化に使用)
    "tune_params": True,           # パラメータチューニング、lgb_hopt,xgb_hopt,nn_hopt,False
    "target_encoder": None, #TargetEncodingBlock(use_cache=False),     # target encodingをしない場合はNone
}
cv_setting = {
    "key_cols": ["datetime", "station_id"],
    "target_col": "bikes_available",
    "initial_fold_date": "2014-09-01",
}

In [173]:
memo = "LSTMのマルチステップ予測"
runner = TimeseriesModelRunner(run_name, model_LSTM_mult, model_params, main_df_new, run_setting, cv_setting, logger, memo)

In [174]:
runner.run_train_cv()

[2024-10-08 22:21:04] - lstm_mult_202410082220 - start training cv
[2024-10-08 22:21:04] - lstm_mult_202410082220 fold 2014-09-01 - start training
[2024-10-08 22:21:35] - Epoch [1/10], Loss: 0.0112
[2024-10-08 22:21:38] - Epoch [2/10], Loss: 0.0093
[2024-10-08 22:21:43] - Epoch [3/10], Loss: 0.0063
[2024-10-08 22:21:46] - Epoch [4/10], Loss: 0.0133
[2024-10-08 22:21:49] - Epoch [5/10], Loss: 0.0052
[2024-10-08 22:21:52] - Epoch [6/10], Loss: 0.0111
[2024-10-08 22:21:55] - Epoch [7/10], Loss: 0.0048
[2024-10-08 22:21:58] - Epoch [8/10], Loss: 0.0101
[2024-10-08 22:22:00] - Epoch [9/10], Loss: 0.0169
[2024-10-08 22:22:03] - Epoch [10/10], Loss: 0.0112
[2024-10-08 22:22:03] - lstm_mult_202410082220 fold 2014-09-01 - end training
[2024-10-08 22:22:03] - lstm_mult_202410082220 fold 2014-10-01 - start training
[2024-10-08 22:37:55] - Epoch [1/10], Loss: 0.0080
[2024-10-08 22:37:58] - Epoch [2/10], Loss: 0.0083
[2024-10-08 22:38:00] - Epoch [3/10], Loss: 0.0087
[2024-10-08 22:38:03] - Epoch [

In [175]:
runner.run_metric_cv()

[2024-10-09 00:22:39] - lstm_mult_202410082220 - start metric cv
memo: LSTMのマルチステップ予測
name:lstm_mult_202410082220	score:2.6642620016301444	score0:2.7136648627127897	score1:2.9100521036021645	score2:2.560972178493083	score3:2.655300169709401	score4:2.629899038676189	score5:2.427796052946193	score6:2.6687005875500014	score7:2.6195400022897037	score8:2.55844053526885	score9:2.7792435614566173	score10:2.7731996179889173	score11:2.6743353088678203
mean: 2.6642620016301444, std: 0.11871080156736571
[2024-10-09 00:23:03] - output predict : /Users/ishizuka/pyworks/Competitions/signate_mufgcup2024/models/lstm_mult_202410082220/va_pred.pkl
[2024-10-09 00:23:03] - lstm_mult_202410082220 - end metric cv


In [176]:
runner.run_predict_cv()

[2024-10-09 00:23:04] - lstm_mult_202410082220 - start prediction cv
[2024-10-09 00:23:45] - output predict : /Users/ishizuka/pyworks/Competitions/signate_mufgcup2024/models/lstm_mult_202410082220/te_pred.pkl
[2024-10-09 00:23:45] - lstm_mult_202410082220 - end prediction cv


In [177]:
path_va_pred = os.path.join(runner.out_dir_name, "va_pred.pkl")
path_te_pred = os.path.join(runner.out_dir_name, "te_pred.pkl")
df_va_pred = pd.read_pickle(path_va_pred)
df_te_pred = pd.read_pickle(path_te_pred)

In [178]:
df_true = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_status.pkl"))[["datetime", "station_id", "bikes_available"]]

In [166]:
# 正解値、検証値、テスト値をプロット
# station_idごとにプロット
station_id = 2
df_true_ = df_true[df_true["station_id"]==station_id]
df_va_pred_ = df_va_pred[df_va_pred["station_id"]==station_id]
df_te_pred_ = df_te_pred[df_te_pred["station_id"]==station_id]
# データ期間も指定する
start_date = "2015-01-01"
end_date = "2015-01-02"
df_true_ = df_true_[(df_true_["datetime"] >= start_date) & (df_true_["datetime"] <= end_date)]
df_va_pred_ = df_va_pred_[(df_va_pred_["datetime"] >= start_date) & (df_va_pred_["datetime"] <= end_date)]
df_te_pred_ = df_te_pred_[(df_te_pred_["datetime"] >= start_date) & (df_te_pred_["datetime"] <= end_date)]
# プロット
plt.figure(figsize=(20, 5))
plt.plot(df_true_["datetime"], df_true_["bikes_available"], label="true")
plt.plot(df_va_pred_["datetime"], df_va_pred_["predict"], label="va_pred")
plt.plot(df_te_pred_["datetime"], df_te_pred_["predict"], label="te_pred")
plt.legend()
plt.show()

,datetime,station_id,bikes_available
0,2014-09-01 01:00:00,0,14.581677
0,2014-09-01 01:00:00,1,8.751324
0,2014-09-01 01:00:00,2,4.754292
0,2014-09-01 01:00:00,3,7.770417
0,2014-09-01 01:00:00,4,7.751976
...,...,...,...
22,2015-08-29 23:00:00,65,8.103925
22,2015-08-29 23:00:00,66,7.315821
22,2015-08-29 23:00:00,67,6.905207
22,2015-08-29 23:00:00,68,6.017195


# モデル構築

In [13]:
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import math
from statistics import mean
from sklearn.metrics import mean_squared_error

from tqdm import tqdm

from datetime import datetime, timedelta
# from runner import Runner

In [14]:
SEQ_LENGTH = 24  # 過去24時間のデータを使用
N_STEPS = 23  # 23期先を予測


In [19]:
train_df = train_df[["station_id", "datetime", "bikes_available", "weekdays"]]

In [20]:
key_cols = ["station_id", "datetime"]
target_col = ["bikes_available"]
feat_cols = [col for col in train_df.columns.to_list() if col not in key_cols]

In [21]:
train_end_datetime = datetime.strptime("2014-07-01 00:00:00", "%Y-%m-%d %H:%M:%S")
valid_start_datetime = train_end_datetime + timedelta(hours=-N_STEPS)
train, test = train_df[train_df["datetime"]<=train_end_datetime], train_df[train_df["datetime"]>=valid_start_datetime]
print(train.shape)
print(test.shape)
display(train.tail())
display(test.head())

(509110, 4)
(105770, 4)


,station_id,datetime,bikes_available,weekdays
509105,65,2014-07-01,13.0,1
509106,66,2014-07-01,8.0,1
509107,67,2014-07-01,1.0,1
509108,68,2014-07-01,7.0,1
509109,69,2014-07-01,5.0,1


,station_id,datetime,bikes_available,weekdays
507430,0,2014-06-30 01:00:00,11.0,0
507431,1,2014-06-30 01:00:00,13.0,0
507432,2,2014-06-30 01:00:00,6.0,0
507433,3,2014-06-30 01:00:00,8.0,0
507434,4,2014-06-30 01:00:00,3.0,0


In [22]:
# staion_id,datetime以外のカラムをデータの正規化
train_ = train.copy()
scale_cols = target_col + feat_cols
scaler = MinMaxScaler()
scaler_for_inverse = MinMaxScaler(feature_range=(0, 1))
train[scale_cols] = scaler.fit_transform(train[scale_cols])
bikes_available_scale = scaler_for_inverse.fit_transform(train_[["bikes_available"]])
print(train.shape)

(509110, 4)


/var/folders/qk/pwcsrt352q5ds79rrtgfbtrh0000gn/T/ipykernel_12988/3530751440.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train[scale_cols] = scaler.fit_transform(train[scale_cols])


In [23]:
train.tail()

,station_id,datetime,bikes_available,weekdays
509105,65,2014-07-01,0.481481,0.166667
509106,66,2014-07-01,0.296296,0.166667
509107,67,2014-07-01,0.037037,0.166667
509108,68,2014-07-01,0.259259,0.166667
509109,69,2014-07-01,0.185185,0.166667


In [36]:
# データのサンプルとして、n日00:00までのデータを使ってn日01:00~23:00の予測を行う準備
def create_sequences_for_forecast(data, feature_columns, seq_length, n_steps):
    list_x, list_y = [], []
    list_key = []
    for station_id in data['station_id'].unique():
        station_data = data[data['station_id'] == station_id]
        station_data = station_data.set_index('datetime')
        # 00:00までのデータを取得（1日分前のデータを想定）
        for day in station_data.index.normalize().unique():
            day = pd.to_datetime(day)
            day_data = station_data.loc[:day+pd.Timedelta(hours=0)]  # n日00:00までのデータ
            if len(day_data) >= seq_length:  # 過去データがシーケンス長よりも多い場合
                x = day_data[feature_columns].values[-seq_length:]  # シーケンス長分のデータを入力
                next_day_data = station_data.loc[day+pd.Timedelta(hours=1): day+pd.Timedelta(hours=23)]
                if len(next_day_data) == n_steps:  # 予測対象の23時間分が揃っている場合
                    y = next_day_data['bikes_available'].values
                    list_x.append(x)
                    list_y.append(y)
                    list_key.append(next_day_data.reset_index()[key_cols])
    return np.array(list_x), np.array(list_y), pd.concat(list_key, axis=0)

In [37]:
# シーケンスの作成
# 並び替え
train = train.sort_values(key_cols).reset_index(drop=True)
train_X, train_y, train_key = create_sequences_for_forecast(train, feat_cols, seq_length=SEQ_LENGTH, n_steps=N_STEPS)
print(train_X.shape, train_y.shape)

(21140, 24, 2) (21140, 23)


In [42]:
train_key

,station_id,datetime
0,0,2013-09-02 01:00:00
1,0,2013-09-02 02:00:00
2,0,2013-09-02 03:00:00
3,0,2013-09-02 04:00:00
4,0,2013-09-02 05:00:00
...,...,...
18,69,2014-06-30 19:00:00
19,69,2014-06-30 20:00:00
20,69,2014-06-30 21:00:00
21,69,2014-06-30 22:00:00


In [43]:
# tensorに変換
train_X_tensor = torch.tensor(train_X, dtype=torch.float32)
train_y_tensor = torch.tensor(train_y, dtype=torch.float32)

# データローダーを作成
train_dataset = TensorDataset(train_X_tensor, train_y_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [44]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])  # 最後のタイムステップの出力
        return out

# モデルパラメータの設定
input_size = train_X_tensor.shape[2]  # 入力の次元
hidden_size = 64
output_size = N_STEPS  # 23期先の予測
num_epochs = 20
learning_rate = 0.001

# モデルの初期化
model = LSTMModel(input_size, hidden_size, output_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [45]:
# トレーニングループ
for epoch in range(num_epochs):
    model.train()
    for i, (inputs, targets) in enumerate(train_loader):
    
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/20], Loss: 0.0078
Epoch [2/20], Loss: 0.0055
Epoch [3/20], Loss: 0.0074
Epoch [4/20], Loss: 0.0095
Epoch [5/20], Loss: 0.0070
Epoch [6/20], Loss: 0.0094
Epoch [7/20], Loss: 0.0083
Epoch [8/20], Loss: 0.0080
Epoch [9/20], Loss: 0.0102
Epoch [10/20], Loss: 0.0035
Epoch [11/20], Loss: 0.0111
Epoch [12/20], Loss: 0.0129
Epoch [13/20], Loss: 0.0083
Epoch [14/20], Loss: 0.0098
Epoch [15/20], Loss: 0.0096
Epoch [16/20], Loss: 0.0045
Epoch [17/20], Loss: 0.0065
Epoch [18/20], Loss: 0.0063
Epoch [19/20], Loss: 0.0090
Epoch [20/20], Loss: 0.0068


In [48]:
# テストデータの作成
test[scale_cols] = scaler.fit_transform(test[scale_cols])

# 並び替え
test = test.sort_values(key_cols).reset_index(drop=True)

# シーケンスの作成
test_X, test_y, test_key = create_sequences_for_forecast(test, feat_cols, seq_length=SEQ_LENGTH, n_steps=N_STEPS)

# テンソルに変換
test_X_tensor = torch.tensor(test_X, dtype=torch.float32)
test_y_tensor = torch.tensor(test_y, dtype=torch.float32)

print(test_X_tensor.shape, test_y_tensor.shape)

torch.Size([4340, 24, 2]) torch.Size([4340, 23])


In [49]:
test_key

,station_id,datetime
0,0,2014-07-01 01:00:00
1,0,2014-07-01 02:00:00
2,0,2014-07-01 03:00:00
3,0,2014-07-01 04:00:00
4,0,2014-07-01 05:00:00
...,...,...
18,69,2014-08-31 19:00:00
19,69,2014-08-31 20:00:00
20,69,2014-08-31 21:00:00
21,69,2014-08-31 22:00:00


In [50]:
# 予測
model.eval()
with torch.no_grad():
    train_outputs = model(train_X_tensor)
    test_outputs = model(test_X_tensor)

# tensorをnumpyに変換
train_pred = train_outputs.numpy()
test_pred = test_outputs.numpy()
train_true = train_y_tensor.numpy()
test_true = test_y_tensor.numpy()

#スケールをもとに戻す
train_predict = scaler_for_inverse.inverse_transform(train_pred)
train_true = scaler_for_inverse.inverse_transform(train_true)
test_predict = scaler_for_inverse.inverse_transform(test_pred)
test_true = scaler_for_inverse.inverse_transform(test_true)

#各ステーションのスコアの平均値を算出
train_score_list = []
test_score_list = []
num_per_station_tarin = int(len(train_pred) / 70)
num_per_station_test = int(len(test_pred) / 70)
for i in range(70):
    train_score = math.sqrt(mean_squared_error(train_true[i*num_per_station_tarin:(i+1)*num_per_station_tarin], train_predict[i*num_per_station_tarin:(i+1)*num_per_station_tarin]))
    train_score_list.append(train_score)
    test_score = math.sqrt(mean_squared_error(test_true[i*num_per_station_test:(i+1)*num_per_station_test], test_predict[i*num_per_station_test:(i+1)*num_per_station_test]))
    test_score_list.append(test_score)
    
print("trainのRMSE平均 : ",mean(train_score_list))
print("testのRMSE平均 : ",mean(test_score_list))

trainのRMSE平均 :  2.1620084146228753
testのRMSE平均 :  2.3159332197447013


In [64]:
test_key["pred"] = test_predict.reshape(-1, 1)

In [67]:
test_key["true"] = test_true.reshape(-1, 1)

In [68]:
test_key

,station_id,datetime,pred,true
0,0,2014-07-01 01:00:00,11.865046,11.0
1,0,2014-07-01 02:00:00,11.964834,11.0
2,0,2014-07-01 03:00:00,12.083977,11.0
3,0,2014-07-01 04:00:00,12.022257,12.0
4,0,2014-07-01 05:00:00,12.015991,11.0
...,...,...,...,...
18,69,2014-08-31 19:00:00,9.425167,10.0
19,69,2014-08-31 20:00:00,9.322835,10.0
20,69,2014-08-31 21:00:00,9.377410,10.0
21,69,2014-08-31 22:00:00,9.400207,10.0


In [73]:
pd.merge(test_key, df_status, on=["station_id", "datetime"], how="left").head(50)

,station_id,datetime,pred,true,id,year,month,day,hour,bikes_available,predict
0,0,2014-07-01 01:00:00,11.865046,11.0,7273,2014,7,1,1,11.0,0
1,0,2014-07-01 02:00:00,11.964834,11.0,7274,2014,7,1,2,11.0,0
2,0,2014-07-01 03:00:00,12.083977,11.0,7275,2014,7,1,3,11.0,0
3,0,2014-07-01 04:00:00,12.022257,12.0,7276,2014,7,1,4,12.0,0
4,0,2014-07-01 05:00:00,12.015991,11.0,7277,2014,7,1,5,11.0,0
5,0,2014-07-01 06:00:00,12.181674,11.0,7278,2014,7,1,6,11.0,0
6,0,2014-07-01 07:00:00,12.585066,14.0,7279,2014,7,1,7,14.0,0
7,0,2014-07-01 08:00:00,13.131074,17.0,7280,2014,7,1,8,17.0,0
8,0,2014-07-01 09:00:00,14.058600,14.0,7281,2014,7,1,9,14.0,0
9,0,2014-07-01 10:00:00,14.167627,12.0,7282,2014,7,1,10,12.0,0


In [ ]:
# 全ての23時間を使用

In [159]:
# シーケンスを作成する関数 (station_id で分けない)
def create_sequences(data, feature_columns, seq_length, n_steps):
    xs, ys = [], []
    feature_values = data[feature_columns].values
    target_values = data[target_col].values
    for i in tqdm(range(len(feature_values) - seq_length - n_steps)):
        x = feature_values[i:i + seq_length]
        y = target_values[i + seq_length:i + seq_length + n_steps]
        xs.append(x)
        ys.append(y)
    xs = np.array(xs)
    ys = np.array(ys)
    # 2次元配列に変換
    ys = ys.reshape(-1, n_steps)
    return xs, ys

In [160]:
# 並び替え
train = train.sort_values(key_cols).reset_index(drop=True)

# シーケンスの作成
train_X, train_y = create_sequences(train, feat_cols, seq_length=SEQ_LENGTH, n_steps=N_STEPS)
print(train_X.shape, train_y.shape)

100%|██████████| 509063/509063 [00:00<00:00, 903465.69it/s]


(509063, 24, 2) (509063, 23)


In [161]:
train_X_tensor = torch.tensor(train_X, dtype=torch.float32)
train_y_tensor = torch.tensor(train_y, dtype=torch.float32)

# データローダーを作成
train_dataset = TensorDataset(train_X_tensor, train_y_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

In [162]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])  # 最後のタイムステップの出力
        return out

# モデルパラメータの設定
input_size = train_X_tensor.shape[2]  # 入力の次元
hidden_size = 64
output_size = N_STEPS  # 23期先の予測
num_epochs = 20
learning_rate = 0.001

# モデルの初期化
model = LSTMModel(input_size, hidden_size, output_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [163]:
# トレーニングループ
for epoch in range(num_epochs):
    model.train()
    for i, (inputs, targets) in enumerate(train_loader):
    
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/20], Loss: 0.0097
Epoch [2/20], Loss: 0.0086
Epoch [3/20], Loss: 0.0110
Epoch [4/20], Loss: 0.0118
Epoch [5/20], Loss: 0.0086
Epoch [6/20], Loss: 0.0104
Epoch [7/20], Loss: 0.0101
Epoch [8/20], Loss: 0.0108
Epoch [9/20], Loss: 0.0095
Epoch [10/20], Loss: 0.0099
Epoch [11/20], Loss: 0.0080
Epoch [12/20], Loss: 0.0122
Epoch [13/20], Loss: 0.0092
Epoch [14/20], Loss: 0.0082
Epoch [15/20], Loss: 0.0125
Epoch [16/20], Loss: 0.0094
Epoch [17/20], Loss: 0.0088
Epoch [18/20], Loss: 0.0090
Epoch [19/20], Loss: 0.0086
Epoch [20/20], Loss: 0.0085


In [164]:
# テストデータの作成
test[scale_cols] = scaler.fit_transform(test[scale_cols])

# 並び替え
train = train.sort_values(key_cols).reset_index(drop=True)
test = test.sort_values(key_cols).reset_index(drop=True)

# シーケンスの作成
train_X, train_y = create_sequences_for_forecast(train, feat_cols, seq_length=SEQ_LENGTH, n_steps=N_STEPS)
test_X, test_y = create_sequences_for_forecast(test, feat_cols, seq_length=SEQ_LENGTH, n_steps=N_STEPS)

# テンソルに変換
train_X_tensor = torch.tensor(train_X, dtype=torch.float32)
train_y_tensor = torch.tensor(train_y, dtype=torch.float32)
test_X_tensor = torch.tensor(test_X, dtype=torch.float32)
test_y_tensor = torch.tensor(test_y, dtype=torch.float32)

In [165]:
# 予測
model.eval()
with torch.no_grad():
    train_outputs = model(train_X_tensor)
    test_outputs = model(test_X_tensor)

# tensorをnumpyに変換
train_pred = train_outputs.numpy()
test_pred = test_outputs.numpy()
train_true = train_y_tensor.numpy()
test_true = test_y_tensor.numpy()

#スケールをもとに戻す
train_predict = scaler_for_inverse.inverse_transform(train_pred)
train_true = scaler_for_inverse.inverse_transform(train_true)
test_predict = scaler_for_inverse.inverse_transform(test_pred)
test_true = scaler_for_inverse.inverse_transform(test_true)

#各ステーションのスコアの平均値を算出
train_score_list = []
test_score_list = []
num_per_station_tarin = int(len(train_pred) / 70)
num_per_station_test = int(len(test_pred) / 70)
for i in range(70):
    train_score = math.sqrt(mean_squared_error(train_true[i*num_per_station_tarin:(i+1)*num_per_station_tarin], train_predict[i*num_per_station_tarin:(i+1)*num_per_station_tarin]))
    train_score_list.append(train_score)
    test_score = math.sqrt(mean_squared_error(test_true[i*num_per_station_test:(i+1)*num_per_station_test], test_predict[i*num_per_station_test:(i+1)*num_per_station_test]))
    test_score_list.append(test_score)
    
print("trainのRMSE平均 : ",mean(train_score_list))
print("testのRMSE平均 : ",mean(test_score_list))

trainのRMSE平均 :  2.131590655223754
testのRMSE平均 :  2.294870317954236


In [106]:
test_ = test[test["datetime"]>="2014-07-01"]
list_test_date = test_["datetime"].apply(lambda x: x.date()).unique().tolist()
for date in list_test_date:
    datetime = pd.to_datetime(date)
    test_ = test[test["datetime"].apply(lambda x: x.date())<=date]

2014-07-01 00:00:00
2014-07-02 00:00:00
2014-07-03 00:00:00
2014-07-04 00:00:00
2014-07-05 00:00:00
2014-07-06 00:00:00
2014-07-07 00:00:00
2014-07-08 00:00:00
2014-07-09 00:00:00
2014-07-10 00:00:00
2014-07-11 00:00:00
2014-07-12 00:00:00
2014-07-13 00:00:00
2014-07-14 00:00:00
2014-07-15 00:00:00
2014-07-16 00:00:00
2014-07-17 00:00:00
2014-07-18 00:00:00
2014-07-19 00:00:00
2014-07-20 00:00:00
2014-07-21 00:00:00
2014-07-22 00:00:00
2014-07-23 00:00:00
2014-07-24 00:00:00
2014-07-25 00:00:00
2014-07-26 00:00:00
2014-07-27 00:00:00
2014-07-28 00:00:00
2014-07-29 00:00:00
2014-07-30 00:00:00
2014-07-31 00:00:00
2014-08-01 00:00:00
2014-08-02 00:00:00
2014-08-03 00:00:00
2014-08-04 00:00:00
2014-08-05 00:00:00
2014-08-06 00:00:00
2014-08-07 00:00:00
2014-08-08 00:00:00
2014-08-09 00:00:00
2014-08-10 00:00:00
2014-08-11 00:00:00
2014-08-12 00:00:00
2014-08-13 00:00:00
2014-08-14 00:00:00
2014-08-15 00:00:00
2014-08-16 00:00:00
2014-08-17 00:00:00
2014-08-18 00:00:00
2014-08-19 00:00:00


In [86]:
list_test_date

[datetime.date(2014, 6, 30),
 datetime.date(2014, 7, 1),
 datetime.date(2014, 7, 2),
 datetime.date(2014, 7, 3),
 datetime.date(2014, 7, 4),
 datetime.date(2014, 7, 5),
 datetime.date(2014, 7, 6),
 datetime.date(2014, 7, 7),
 datetime.date(2014, 7, 8),
 datetime.date(2014, 7, 9),
 datetime.date(2014, 7, 10),
 datetime.date(2014, 7, 11),
 datetime.date(2014, 7, 12),
 datetime.date(2014, 7, 13),
 datetime.date(2014, 7, 14),
 datetime.date(2014, 7, 15),
 datetime.date(2014, 7, 16),
 datetime.date(2014, 7, 17),
 datetime.date(2014, 7, 18),
 datetime.date(2014, 7, 19),
 datetime.date(2014, 7, 20),
 datetime.date(2014, 7, 21),
 datetime.date(2014, 7, 22),
 datetime.date(2014, 7, 23),
 datetime.date(2014, 7, 24),
 datetime.date(2014, 7, 25),
 datetime.date(2014, 7, 26),
 datetime.date(2014, 7, 27),
 datetime.date(2014, 7, 28),
 datetime.date(2014, 7, 29),
 datetime.date(2014, 7, 30),
 datetime.date(2014, 7, 31),
 datetime.date(2014, 8, 1),
 datetime.date(2014, 8, 2),
 datetime.date(2014, 8, 3

In [69]:
# テストデータで予測
model.eval()
preds = []
targets = []
for inputs, target in test_loader:
    outputs = model(inputs)
    preds.append(outputs.detach().numpy())
    targets.append(target.detach().numpy())
preds = np.concatenate(preds, axis=0)
targets = np.concatenate(targets, axis=0)

In [75]:
preds.shape

(105723, 23)

In [73]:
test[test["datetime"]>=train_end_datetime+timedelta(hours=1)]

,station_id,datetime,bikes_available,weekdays
24,0,2014-07-01 01:00:00,0.407407,0.166667
25,0,2014-07-01 02:00:00,0.407407,0.166667
26,0,2014-07-01 03:00:00,0.407407,0.166667
27,0,2014-07-01 04:00:00,0.444444,0.166667
28,0,2014-07-01 05:00:00,0.407407,0.166667
...,...,...,...,...
105765,69,2014-08-31 19:00:00,0.370370,1.000000
105766,69,2014-08-31 20:00:00,0.370370,1.000000
105767,69,2014-08-31 21:00:00,0.370370,1.000000
105768,69,2014-08-31 22:00:00,0.370370,1.000000


In [48]:
def create_dataset(dataset, window_size):
    # station_idごとにLSTMの入力データを作成
    X, Y = [], []
    for station_id in dataset["station_id"].unique():
        temp_df = dataset[dataset["station_id"]==station_id]
        for i in range(len(temp_df)-window_size):
            X.append(temp_df[scale_cols].iloc[i:i+window_size].values)
            Y.append(temp_df["bikes_available"].iloc[i+window_size])
    X = np.array(X)
    Y = np.array(Y)
    return X, Y

In [54]:
trainX, trainY = create_dataset(train, 24)
testX, testY = create_dataset(test, 24)
print(trainX.shape)
print(trainY.shape)

KeyboardInterrupt: 

In [52]:
trainX

array([[[0.40740741, 1.        ],
        [0.40740741, 1.        ],
        [0.40740741, 1.        ],
        ...,
        [0.44444444, 1.        ],
        [0.40740741, 1.        ],
        [0.40740741, 1.        ]],

       [[0.40740741, 1.        ],
        [0.40740741, 1.        ],
        [0.40740741, 1.        ],
        ...,
        [0.40740741, 1.        ],
        [0.40740741, 1.        ],
        [0.44444444, 0.        ]],

       [[0.40740741, 1.        ],
        [0.40740741, 1.        ],
        [0.40740741, 1.        ],
        ...,
        [0.40740741, 1.        ],
        [0.44444444, 0.        ],
        [0.40740741, 0.        ]],

       ...,

       [[0.2962963 , 1.        ],
        [0.2962963 , 1.        ],
        [0.2962963 , 1.        ],
        ...,
        [0.14814815, 0.        ],
        [0.14814815, 0.        ],
        [0.18518519, 0.        ]],

       [[0.2962963 , 1.        ],
        [0.2962963 , 1.        ],
        [0.2962963 , 0.        ],
        .

In [ ]:
trainX, trainY = create_dataset(train)
testX, testY = create_dataset(test)
print(trainX.shape)
print(trainY.shape)

In [148]:
def create_dataset(dataset):
    dataX = []
    dataY = np.array([])
    #1680で1つのデータセットであるため、余りの分は使わない
    extra_num = len(dataset) % 70
    max_len = len(dataset)-extra_num
    for i in range(1680,max_len,70):
        xset = []
        for j in range(dataset.shape[1]):
            a = dataset[i-1680:i, j]
            xset.append(a)
        temp_array = np.array(dataset[i:i+70,0])
        dataY = np.concatenate([dataY,temp_array])
        dataX.append(xset)
    dataY = dataY.reshape(-1,70)
    return np.array(dataX), dataY 

In [149]:
trainX, trainY = create_dataset(train_scale)
testX, testY = create_dataset(test_scale)
print(trainX.shape)
print(trainY.shape)

(6984, 2, 1680)
(6984, 70)


In [49]:
# データの形状を確認し、PyTorchのテンソルに変換
trainX_tensor = torch.tensor(trainX, dtype=torch.float32)
trainY_tensor = torch.tensor(trainY, dtype=torch.float32)

# データローダーを作成
train_dataset = TensorDataset(trainX_tensor, trainY_tensor)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# LSTMモデルを定義
class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=1680, hidden_size=50, batch_first=True)
        self.fc = nn.Linear(50, 70)
    
    def forward(self, x):
        h_0 = torch.zeros(1, x.size(0), 50).to(x.device)
        c_0 = torch.zeros(1, x.size(0), 50).to(x.device)
        out, _ = self.lstm(x, (h_0, c_0))
        out = self.fc(out[:, -1, :])
        return out

# モデルのインスタンスを作成
model_pt = LSTMModel()

# 損失関数とオプティマイザを定義
criterion = nn.MSELoss()
optimizer = optim.Adam(model_pt.parameters(), lr=0.001)

# モデルをトレーニング
num_epochs = 20
for epoch in range(num_epochs):
    model_pt.train()
    for i, (inputs, targets) in enumerate(train_loader):
        outputs = model_pt(inputs)
        loss = criterion(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/20], Loss: 0.0160
Epoch [2/20], Loss: 0.0098
Epoch [3/20], Loss: 0.0151
Epoch [4/20], Loss: 0.0101
Epoch [5/20], Loss: 0.0131
Epoch [6/20], Loss: 0.0043
Epoch [7/20], Loss: 0.0107
Epoch [8/20], Loss: 0.0067
Epoch [9/20], Loss: 0.0136
Epoch [10/20], Loss: 0.0084
Epoch [11/20], Loss: 0.0109
Epoch [12/20], Loss: 0.0117
Epoch [13/20], Loss: 0.0104
Epoch [14/20], Loss: 0.0081
Epoch [15/20], Loss: 0.0070
Epoch [16/20], Loss: 0.0137
Epoch [17/20], Loss: 0.0125
Epoch [18/20], Loss: 0.0158
Epoch [19/20], Loss: 0.0093
Epoch [20/20], Loss: 0.0046


In [66]:
# 
#学習済みモデルで予測
model = model_pt
model.eval()
trainX_tensor = torch.tensor(trainX, dtype=torch.float32)
testX_tensor = torch.tensor(testX, dtype=torch.float32)
train_predict = model(trainX_tensor)
test_predict = model(testX_tensor)

#スケールをもとに戻す
train_predict = scaler_for_inverse.inverse_transform(train_predict.detach().numpy())
trainY = scaler_for_inverse.inverse_transform(trainY)
test_predict = scaler_for_inverse.inverse_transform(test_predict.detach().numpy())
testY = scaler_for_inverse.inverse_transform(testY)

#各ステーションのスコアの平均値を算出
train_score_list = []
test_score_list = []
for i in range(70):
    trainscore = math.sqrt(mean_squared_error(trainY[:,i], train_predict[:,i]))
    train_score_list.append(trainscore)
    testscore = math.sqrt(mean_squared_error(testY[:,i], test_predict[:,i]))
    test_score_list.append(testscore)
    
print("trainのRMSE平均 : ",mean(train_score_list))
print("testのRMSE平均 : ",mean(test_score_list))

trainのRMSE平均 :  2.5844546330388116
testのRMSE平均 :  3.051291571389852


In [86]:
from datetime import timedelta

In [ ]:
# 2. モデルインターフェース例
class ForecastingModelBase:
    def __init__(self, model):
        self.model = model

    @abstractmethod
    def predict(self, X: pd.DataFrame) -> np.array:
        """
        1期先の予測を行う関数。
        """
        return self.model.predict(X)
    
class LSTM(ForecastingModelBase):
    def __init__(self, model):
        super().__init__(model)

    def predict(self, X: pd.DataFrame) -> np.array:
        """
        LSTMモデルを使用して1期先の予測を行う関数。
        """
        return self.model.predict(X)

In [ ]:
# 1. 特徴量生成パイプラインの例
def feature_generation_pipeline(df_station: pd.DataFrame, df_status: pd.DataFrame, df_trip: pd.DataFrame, df_weather: pd.DataFrame, target_date: pd.Timestamp) -> pd.DataFrame:
    """
    各種データから特徴量を生成するパイプライン。
    target_dateのデータまでを使用し、ラグ特徴量などを生成する例。
    """
    df_status["weekdays"] = df_status["datetime"].dt.weekday
    
    return df_station, df_status, df_trip, df_weather

# 2. モデルインターフェース例
class ForecastingModelBase:
    def __init__(self, model):
        self.model = model

    @abstractmethod
    def predict(self, X: pd.DataFrame) -> np.array:
        """
        1期先の予測を行う関数。
        """
        return self.model.predict(X)
    
class LSTM(ForecastingModelBase):
    def __init__(self, model):
        super().__init__(model)

    def predict(self, X: pd.DataFrame) -> np.array:
        """
        LSTMモデルを使用して1期先の予測を行う関数。
        """
        return self.model.predict(X)

# 3. 再帰的な予測を行う関数
def recursive_forecasting(df_station, df_status, df_trip, df_weather, target_date, model: ForecastingModel, steps: int = 23) -> pd.DataFrame:
    """
    再帰的に特徴量を生成し、1期先から23期先までの予測を行う関数。
    
    df_station: 駅情報のデータフレーム
    df_status: 駅のバイク利用状況のデータフレーム
    df_trip: トリップデータのデータフレーム
    df_weather: 天気データのデータフレーム
    target_date: 予測対象日（target_dateの0時までのデータが使用可能）
    model: 予測モデル (ForecastingModel型)
    steps: 予測する期数 (デフォルトは23期先)
    
    return: 1期先から23期先までの予測結果を含むDataFrame
    """
    
    # 過去データで初期の特徴量生成
    df_station, df_status, df_trip, df_weather = feature_generation_pipeline(df_station, df_status, df_trip, df_weather, target_date)

    # 予測結果を格納するリスト
    predictions = []

    # 再帰的に1期先ずつ予測を行うループ
    for step in range(1, 24):
        
        # モデルで1期先の予測を行う
        model = LSTM(model_pt)
        next_pred = model.predict(df_station, df_status, df_trip, df_weather)
        
        # 予測結果を次のステップの特徴量生成に使用
        next_status = pd.DataFrame(columns=['station_id', 'bikes_available', 'datetime'])
        next_status['bikes_available'] = next_pred
        next_status['station_id'] = [x for x in range(0,70)]
        next_status['datetime'] = target_date + timedelta(hours=step)

        # 予測結果を保存
        predictions.append(next_status)
        
        # 次の期の特徴量を生成（新しい予測値を加える）
        df_status = pd.concat([df_status, next_status], axis=0)
        df_status.sort_values(['datetime', 'station_id'], inplace=True)
        df_station, df_status, df_trip, df_weather = feature_generation_pipeline(df_station, df_status, df_trip, df_weather, target_date)

    # 予測結果をDataFrameとして返す
    return pd.concat(predictions, axis=0)

In [ ]:
# マルチステップモデル

In [88]:
train_df

,bikes_available,weekdays
0,11.0,6
1,8.0,6
2,5.0,6
3,9.0,6
4,8.0,6
...,...,...
613195,10.0,6
613196,9.0,6
613197,7.0,6
613198,8.0,6


In [89]:
evaluation_dataset_df

,datetime,year,month,day,hour,station_id,bikes_available,predict,weekdays
0,2014-08-31 00:00:00,2014,8,31,0,0,11.0,0,6
1,2014-08-31 00:00:00,2014,8,31,0,1,9.0,0,6
2,2014-08-31 00:00:00,2014,8,31,0,2,4.0,0,6
3,2014-08-31 00:00:00,2014,8,31,0,3,8.0,0,6
4,2014-08-31 00:00:00,2014,8,31,0,4,7.0,0,6
...,...,...,...,...,...,...,...,...,...
614875,2015-08-31 23:00:00,2015,8,31,23,65,13.0,0,0
614876,2015-08-31 23:00:00,2015,8,31,23,66,7.0,0,0
614877,2015-08-31 23:00:00,2015,8,31,23,67,6.0,0,0
614878,2015-08-31 23:00:00,2015,8,31,23,68,5.0,0,0


In [108]:
length = len(train_df)
train_size = int(length * 0.8)
test_size = length - train_size
train, test = train_df.iloc[0:train_size,:], train_df.iloc[train_size:length,:]

# データをスケーリング
scaler = MinMaxScaler(feature_range=(0, 1))
train_df_scale = scaler.fit_transform(train)
test_df_scale = scaler.transform(test)

# 入力データ (X) とターゲットデータ (y) を作成
def create_sequences(data, seq_length=24, pred_length=23):
    X, y = [], []
    for i in range(len(data) - seq_length - pred_length):
        X.append(data[i:i+seq_length])  # 特徴量
        y.append(data[i+seq_length:i+seq_length+pred_length, -1])  # bikes_available の未来の値
    return np.array(X), np.array(y)

seq_length = 24  # 24時間の履歴
pred_length = 23  # 23時間先の予測
train_X, train_y = create_sequences(train_df_scale, seq_length, pred_length)
test_X, test_y = create_sequences(test_df_scale, seq_length, pred_length)

In [109]:
train_X.shape, train_y.shape

((490513, 24, 2), (490513, 23))

In [110]:
test_X.shape, test_y.shape

((122593, 24, 2), (122593, 23))

In [120]:
train_dataset_df[train_dataset_df["station_id"]==0]

,datetime,year,month,day,hour,station_id,bikes_available,predict,weekdays
0,2013-09-01 00:00:00,2013,9,1,0,0,11.0,0,6
1,2013-09-01 01:00:00,2013,9,1,1,0,11.0,0,6
2,2013-09-01 02:00:00,2013,9,1,2,0,11.0,0,6
3,2013-09-01 03:00:00,2013,9,1,3,0,11.0,0,6
4,2013-09-01 04:00:00,2013,9,1,4,0,11.0,0,6
...,...,...,...,...,...,...,...,...,...
8755,2014-08-31 19:00:00,2014,8,31,19,0,14.0,0,6
8756,2014-08-31 20:00:00,2014,8,31,20,0,15.0,0,6
8757,2014-08-31 21:00:00,2014,8,31,21,0,15.0,0,6
8758,2014-08-31 22:00:00,2014,8,31,22,0,15.0,0,6


(122593, 23)

１〜２３期先の予測が必要
- bikes_availableを時系列予測

In [ ]:
# i